# 08 — Register Artifacts / ลงทะเบียน artifact กลับเข้า repo

**EN.** Verifies the v3 layout, refreshes `choice_matrix.json`, validates that
the registry sidecar does **not** clobber the backend's `bundle.json` (per
`CLAUDE.md`), prints the exact `git`/`git lfs` commands to push the artifacts
back to the repo from a local machine, and writes `MODEL_CARD.md` if missing.

**TH.** ตรวจ layout v3, อัปเดต `choice_matrix.json`, ยืนยันว่า `registry.json`
ไม่ทับ `bundle.json` ของ backend (ตาม `CLAUDE.md`), แสดงคำสั่ง `git` / `git lfs`
ที่ใช้ push artifact ขึ้น repo จากเครื่อง local, และเขียน `MODEL_CARD.md`
ถ้ายังไม่มี.


## 1. Setup / ตั้งค่า

In [ ]:
# --- bootstrap ------------------------------------------------------------
import os, sys
REPO_DIR = "/content/Heat-wave-backend"
if not os.path.exists(REPO_DIR):
    !bash {REPO_DIR}/scripts/colab_bootstrap.sh || true
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(os.getcwd())


In [ ]:
# --- imports --------------------------------------------------------------
import json
from datetime import datetime, timezone
from pathlib import Path

from app.data.stations import STATIONS

V3_DIR = Path("app/models/forecast_v3")
HORIZONS = [6, 12, 24, 48, 72]
EXPECTED_BUNDLE = "bundle.json"
EXPECTED_REGISTRY = "registry.json"
OPTIONAL_FILES = ["classifier.json", "calibration.json"]

print("v3 root:", V3_DIR.resolve())
print("stations:", list(STATIONS.keys()))
print("horizons:", HORIZONS)


## 2. Verify v3 layout / ตรวจสอบ layout

For each `(station, horizon)` we expect:

- `bundle.json` — backend-owned metadata (must exist).
- `registry.json` — registry sidecar from `save_model_v3`.
- A regressor head + q05/q50/q95 quantile heads (LightGBM `.txt` boosters
  for `lightgbm_quantile`, or `.ubj` boosters for the XGBoost backend).
- `classifier.json` — danger gate classifier (optional but expected).
- `calibration.json` — bias-correction sidecar from notebook 06 (optional).


In [ ]:
# --- walk the v3 tree and tabulate which files are present ---------------
import pandas as pd

rows = []
for sid in STATIONS:
    for h in HORIZONS:
        slot = V3_DIR / sid / f"h{h}"
        if not slot.exists():
            rows.append({"station": sid, "horizon": h, "exists": False, "files": ""})
            continue
        files = sorted(p.name for p in slot.iterdir() if p.is_file())
        rows.append({
            "station": sid,
            "horizon": h,
            "exists": True,
            "has_bundle": EXPECTED_BUNDLE in files,
            "has_registry": EXPECTED_REGISTRY in files,
            "has_classifier": "classifier.json" in files,
            "has_calibration": "calibration.json" in files,
            "n_boosters": sum(1 for f in files if f.endswith((".txt", ".ubj"))),
            "files": ", ".join(files),
        })
checklist = pd.DataFrame(rows)
display(checklist[["station", "horizon", "exists", "has_bundle", "has_registry",
                   "has_classifier", "has_calibration", "n_boosters"]])

missing = checklist[(~checklist["exists"]) | (~checklist.get("has_bundle", False))]
if len(missing) == 0:
    print("OK — every (station, horizon) slot has a bundle.json")
else:
    print("WARNING — missing slots:")
    display(missing)


## 3. Update `choice_matrix.json` / อัปเดตเมทริกซ์เลือก backend

We pick `lightgbm_quantile` when the bundle's `backend_name` is one of the
LightGBM variants, else fall back to `xgboost`. The matrix is rewritten only
if the inferred mapping differs from disk.


In [ ]:
# --- read each bundle's backend_name and rebuild choice_matrix -----------
matrix_path = V3_DIR / "choice_matrix.json"
existing_matrix = json.loads(matrix_path.read_text()) if matrix_path.exists() else {}

inferred: dict[str, dict[str, str]] = {}
for sid in STATIONS:
    for h in HORIZONS:
        slot = V3_DIR / sid / f"h{h}"
        bundle_path = slot / EXPECTED_BUNDLE
        if not bundle_path.exists():
            continue
        try:
            bundle = json.loads(bundle_path.read_text())
        except Exception as exc:
            print(f"[{sid} h{h}] could not read bundle.json: {exc}")
            continue
        backend = bundle.get("backend_name")
        if not backend:
            # Fallback heuristic: infer from booster files present in slot.
            # LightGBM quantile backend writes per-quantile .txt boosters;
            # hi_quantile backend writes a single .txt booster (direct HI head).
            quantile_files = list(slot.glob("q*.txt")) + list(slot.glob("q*.ubj"))
            hi_txt_files = [p for p in slot.glob("*.txt") if not p.name.startswith("q")]
            if quantile_files:
                backend = "lightgbm_quantile"
            elif hi_txt_files:
                backend = "lightgbm_hi_quantile"
            else:
                # Last resort: flag as unknown so operators fix bundle.json manually
                # rather than silently writing a label that load_latest_v3 rejects.
                print(f"[{sid} h{h}] WARNING: could not infer backend — skipping slot")
                continue
        inferred.setdefault(sid, {})[str(h)] = backend

if inferred == existing_matrix:
    print("choice_matrix.json is already up-to-date.")
else:
    matrix_path.write_text(json.dumps(inferred, indent=2))
    print(f"Rewrote {matrix_path}")
    print(json.dumps(inferred, indent=2))

## 4. Validate metadata / ตรวจสอบ metadata

Per `CLAUDE.md`, the registry sidecar (`registry.json`) MUST NOT overwrite the
backend-owned fields stored in `bundle.json`. We assert that no key written
into `registry.json` collides with a key in `bundle.json` whose value differs.


In [ ]:
# --- detect overlap between bundle.json and registry.json -----------------
issues: list[dict] = []
for sid in STATIONS:
    for h in HORIZONS:
        slot = V3_DIR / sid / f"h{h}"
        bp = slot / EXPECTED_BUNDLE
        rp = slot / EXPECTED_REGISTRY
        if not bp.exists() or not rp.exists():
            continue
        bundle = json.loads(bp.read_text())
        sidecar = json.loads(rp.read_text())
        for k, v in sidecar.items():
            if k in bundle and bundle[k] != v:
                # backend_name / feature_list / target_kind appear in both by
                # design — that's expected and they should match. Anything else
                # overlapping is a bug.
                if k in {"backend_name", "feature_list", "target_kind",
                         "station_id", "horizon_h"}:
                    if bundle[k] != v:
                        issues.append({"station": sid, "horizon": h, "key": k,
                                       "bundle": bundle[k], "registry": v,
                                       "severity": "design-mismatch"})
                else:
                    issues.append({"station": sid, "horizon": h, "key": k,
                                   "bundle": bundle[k], "registry": v,
                                   "severity": "clobber"})
if issues:
    print("Found metadata-overlap issues:")
    display(pd.DataFrame(issues))
else:
    print("OK — registry.json never clobbers backend-owned bundle.json fields.")


## 5. Git push / คำสั่ง git ที่ต้องรัน local

We do NOT have credentials in Colab — these commands are printed for the
operator to run on their local machine after pulling the Drive copy. The
`git lfs track` lines cover both XGBoost binary (`*.ubj`) and LightGBM text
boosters (`*.txt`). Adjust globs if you decide LightGBM `.txt` files are small
enough to commit without LFS.


In [ ]:
# --- print the local commit/push recipe -----------------------------------
print("# 1. Sync from Drive to your local checkout (after `gcloud rsync`/`rclone`):")
print("git lfs install")
print('git lfs track "*.ubj" "*.txt"')
print("git add .gitattributes")
print("git add app/models/forecast_v3/")
print("git add configs/risk/thresholds.yaml runs/")
print('git commit -m "feat(forecast): retrain v3 forecasters $(date -u +%Y-%m-%d)"')
print("git push origin main")
print()
print("# 2. (Optional) dry-run from local to see what would change:")
print("git status --porcelain | head -50")


In [ ]:
# --- optional: live `git status --porcelain` if running on local Linux ----
import shutil, subprocess
git_bin = shutil.which("git")
if git_bin and Path(".git").exists():
    out = subprocess.run([git_bin, "status", "--porcelain"], capture_output=True, text=True)
    head = "\n".join(out.stdout.splitlines()[:50])
    print(head if head else "(working tree clean)")
else:
    print("git not available or not a git repo (running in Colab is normal — skipping).")


## 6. Local smoke test / smoke test บนเครื่อง local

After pulling the artifacts and committing locally, run the slow test suite
to confirm `load_latest_v3` resolves every (station, horizon):

```powershell
# Windows PowerShell (matches CLAUDE.md guidance)
.\.venv\Scripts\Activate.ps1
pytest -m slow tests/                   # full slow suite
pytest -m slow tests/test_forecast.py   # forecast-only sanity
```

```bash
# macOS / Linux
source .venv/bin/activate
pytest -m slow tests/
```

Verify the FastAPI app picks up the new artifacts at startup:

```bash
uvicorn app.main:app --reload
# inspect logs for "Pre-warmed v3 forecaster ... h=24" lines per station.
```


## 7. Model card / เขียน Model Card

`MODEL_CARD.md` is created **only if missing** so we never overwrite hand-edited
content. Edit the file in subsequent runs by hand or delete it before re-running
this cell.


In [ ]:
# --- write MODEL_CARD.md (idempotent: skipped if it exists) ---------------
card_path = V3_DIR / "MODEL_CARD.md"
if card_path.exists():
    print(f"{card_path} already exists — leaving it untouched.")
else:
    today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    discovered_backends = sorted({
        json.loads(p.read_text()).get("backend_name", "?")
        for p in V3_DIR.glob("*/h*/bundle.json")
    })
    n_slots = sum(1 for _ in V3_DIR.glob("*/h*/bundle.json"))
    body = f'''# HeatShield AI — v3 Forecast Model Card / การ์ดโมเดล

_Last updated_ {today}

## Intended use / การใช้งานที่ตั้งใจ

**EN.** Hourly heat-index nowcasts and 6/12/24/48/72-hour forecasts for the
five Thai TMD stations registered in `app/data/stations.py` (BKK_01, CNX_01,
KKN_01, HYI_01, RYG_01). Outputs feed the FastAPI risk endpoint and the
public `risk_level` / action-card surface.

**TH.** ใช้สำหรับพยากรณ์ heat index รายชั่วโมงและล่วงหน้า 6/12/24/48/72
ชั่วโมง สำหรับ 5 สถานี TMD ที่ขึ้นทะเบียนใน `app/data/stations.py`.
ผลลัพธ์ป้อนเข้า FastAPI endpoint สำหรับความเสี่ยงและ action card.

## Data scope / ขอบเขตข้อมูล

- TMD station observations (1–2 y window).
- ERA5 reanalysis at 0.25° (3 y window via CDS API).
- NASA POWER (5 y window, no auth).
- Loader prefers ERA5 > NASA POWER for the same `(station_id, ts_utc)`.

## Backends in use / backend ที่ใช้งานจริง

- {", ".join(discovered_backends) or "(none discovered)"}
- {n_slots} (station, horizon) slots populated.
- Choice matrix lives at `app/models/forecast_v3/choice_matrix.json`.

## Known failure modes / ข้อจำกัดที่ทราบ

- **48 / 72 h horizons are awareness only.** `fuse_risk` flags
  `horizon_type="awareness"` for those horizons; consumers must surface them
  as situational guidance, not operational alerts. — โหมด 48/72h เป็นเพียง
  ข้อมูลเตือนล่วงหน้า ไม่ใช้เป็น operational alert.
- **Low-confidence guard.** When the most recent observation is older than 60
  minutes or the PI width exceeds 4°C (or 1.6× the v3 median PI width), the
  pipeline marks the prediction `low_confidence`. — มี guard ที่ตั้งสถานะ
  low_confidence อัตโนมัติ.
- **Closed station registry.** Only the five Thai TMD stations are supported.
  Adding a new station requires updating `app/data/stations.py` and rerunning
  ingest + training. — ระบบรองรับ 5 สถานีเท่านั้น.
- **Calibration coverage.** A station/horizon without `calibration.json`
  passes through uncalibrated; `fuse_risk` still works but bias may be larger.
  — ถ้าไม่มีไฟล์ calibration.json จะใช้ค่าดิบ.

## Retraining cadence / รอบการ retrain

- **Monthly** full retrain across all stations × horizons (driven by Colab
  notebooks 02–08 in order). — เทรนใหม่ทุกเดือน.
- **Ad-hoc** retrain when station coverage drops below 90% for the trailing
  30 days, or when the regression skill score versus the best baseline turns
  negative (see `07_evaluate.ipynb` overall row).
- After every retrain, the user requires the full evaluation report (metrics
  + all charts) inline before publishing — see `notebooks/colab/07_evaluate.ipynb`.

## Provenance / แหล่งที่มา

- Code: `app/ml/forecast/`, `app/core/risk_fusion.py`,
  `app/core/calibration.py`.
- Configs: `configs/risk/thresholds.yaml`,
  `configs/train/classifier.yaml` (if present).
- Tests: run `pytest -m slow tests/` after registration.
'''
    card_path.write_text(body, encoding="utf-8")
    print(f"wrote {card_path} ({len(body)} chars)")


### End of 08_register / จบการลงทะเบียน

**EN.** All artifacts have been verified, the backend choice matrix is current,
metadata sidecars are conflict-free, and the model card is in place. Run the
printed `git` / `git lfs` commands locally to publish.

**TH.** Artifact ผ่านการตรวจครบ, choice matrix อัปเดตแล้ว, ไม่มี metadata ซ้อนทับ,
และ model card พร้อมใช้งาน. ใช้คำสั่ง `git` / `git lfs` ที่แสดงไว้บน local
เพื่อ push.
